In [24]:
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from scipy.spatial.distance import cdist

from sklearn.cluster import KMeans
from sklearn.cluster import MiniBatchKMeans
from sklearn.cluster import BisectingKMeans
from sklearn.cluster import Birch
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN

from sklearn.pipeline import (
    Pipeline,
    TransformerMixin
)

from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder
)

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    StratifiedKFold,
    ShuffleSplit
)

from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import GridSearchCV

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.data import dicionario as dic


In [25]:
data_path = Path("../data/processed/hotel_bookings_tratado.csv")
preprocessed_path = Path("../data/processed/hotel_bookings_preprocessed.csv")
dict_path = Path("../data/processed/dicionario.csv")

In [26]:
df = pd.read_csv(data_path, sep=";")
df_preprocessed = pd.read_csv(preprocessed_path, sep=";")

display(
    Markdown(
        f"### Conjunto de dados — "
        f"{len(df):,} reservas × {df.shape[1]} colunas".replace(",", ".")
    )
)
display(df.head())

df_dict = (
    pd.read_csv(dict_path, sep=";")
    .assign(coluna=lambda tabela: tabela["variavel"])
)

display(Markdown("### Dicionário de dados"))
display(df_dict.head())

display(
    Markdown(
        "Tipos declarados: "
        + " · ".join(
            f"**{tipo}** {quantidade}"
            for tipo, quantidade in df_dict["Tipo"].value_counts().items()
        )
    )
)

### Conjunto de dados — 119.390 reservas × 37 colunas

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,total_nights,total_guests,room_changed,has_special_requests
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0.0,0,0,Check-Out,2015-07-01,2015-07-01,0,2.0,0,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0.0,0,0,Check-Out,2015-07-01,2015-07-01,0,2.0,0,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,75.0,0,0,Check-Out,2015-07-02,2015-07-01,1,1.0,1,0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,75.0,0,0,Check-Out,2015-07-02,2015-07-01,1,1.0,0,0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,98.0,0,1,Check-Out,2015-07-03,2015-07-01,2,2.0,0,1


### Dicionário de dados

,variavel,descricao,valores,Tipo,coluna
0,hotel,Tipo de hotel da reserva: City Hotel ou Resort...,"['City Hotel', 'Resort Hotel']",Nominal,hotel
1,is_canceled,Indicador de cancelamento da reserva (1 = canc...,"['0', '1']",Binária,is_canceled
2,lead_time,Número de dias entre a data de reserva e a dat...,"[0, 737]",Discreta,lead_time
3,arrival_date_year,Ano previsto para a chegada,"[2015, 2017]",Discreta,arrival_date_year
4,arrival_date_month,Mês previsto para a chegada,"['April', 'August', 'December', 'February', 'J...",Nominal,arrival_date_month


Tipos declarados: **Discreta** 15 · **Nominal** 13 · **Binária** 2 · **Contínua** 1 · **Data** 1

In [27]:
colunas_derivadas = {
    "arrival_date",
    "total_nights",
    "total_guests",
    "room_changed",
    "has_special_requests",
}

fora_do_dicionario = sorted(
    colunas_base - colunas_dicionario - colunas_derivadas
)

display(
    Markdown(
        f"- colunas derivadas aceitas: **{len(colunas_base & colunas_derivadas)}**\n"
        f"- colunas da base sem descrição: **{len(fora_do_dicionario)}** "
        f"{fora_do_dicionario}\n"
        f"- matriz pré-processada: **{df_preprocessed.shape[0]:,} × "
        f"{df_preprocessed.shape[1]}**".replace(",", ".")
    )
)

assert not fora_do_dicionario

- colunas derivadas aceitas: **5**
- colunas da base sem descrição: **0** []
- matriz pré-processada: **119.390 × 98**

In [28]:
# Selecionando os atributos da base hoteleira

useless_columns = [
    "is_canceled",
    "reservation_status",
    "reservation_status_date",
    "arrival_date",
    "agent",
    "company",
]

def colunas_do_tipo(*tipos):
    return [
        coluna
        for coluna in df_dict.query("Tipo in @tipos")["coluna"].dropna()
        if coluna in df.columns and coluna not in useless_columns
    ]

continuous_columns = colunas_do_tipo(
    "Contínua",
    "Discreta",
    "Binária",
)

ordinal_columns = colunas_do_tipo("Ordinal")
nominal_columns = colunas_do_tipo("Nominal")

derived_binary = [
    "room_changed",
    "has_special_requests",
]

continuous_columns = list(
    dict.fromkeys(continuous_columns + derived_binary)
)

display(
    Markdown(
        f"- contínuas/discretas/binárias: **{len(continuous_columns)}**\n"
        f"- ordinais: **{len(ordinal_columns)}**\n"
        f"- nominais: **{len(nominal_columns)}**\n"
        f"- fora da modelagem: **{len(useless_columns)}**"
    )
)

- contínuas/discretas/binárias: **19**
- ordinais: **0**
- nominais: **10**
- fora da modelagem: **6**

In [29]:
continuous_preprocessor = Pipeline(steps=[
    ('normalization', MinMaxScaler()) # Normalização
])

ordinal_preprocessor = Pipeline(steps=[
    ('normalization', MinMaxScaler()) # Normalização — preserva a ordem e iguala o peso
])

nominal_preprocessor = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)), # Codificação de variáveis
])

preprocessor = ColumnTransformer(transformers=[
    ('continuous', continuous_preprocessor, continuous_columns),
    ('ordinal', ordinal_preprocessor, ordinal_columns),
    ('nominal', nominal_preprocessor, nominal_columns)
])


In [30]:
continuous_preprocessor = Pipeline(
    steps=[
        ("normalization", MinMaxScaler()),
    ]
)

ordinal_preprocessor = Pipeline(
    steps=[
        ("normalization", MinMaxScaler()),
    ]
)

nominal_preprocessor = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", continuous_preprocessor, continuous_columns),
        ("ordinal", ordinal_preprocessor, ordinal_columns),
        ("nominal", nominal_preprocessor, nominal_columns),
    ],
    remainder="drop",
).set_output(transform="pandas")

In [31]:
X = df.drop(columns=useless_columns, errors="ignore")
y = None

X_ref = df_preprocessed.copy()

print(
    f"Matriz pré-processada: "
    f"{X_ref.shape[0]:,} reservas × {X_ref.shape[1]} atributos"
)

Matriz pré-processada: 119,390 reservas × 98 atributos


In [32]:
models = [
    (
        "k-means",
        KMeans(
            n_init=5,
            random_state=42,
        ),
        {
            "n_clusters": [2, 3, 4, 5],
            "init": ["k-means++", "random"],
        },
    ),
    (
        "mini batch k-means",
        MiniBatchKMeans(
            n_init=5,
            batch_size=1024,
            random_state=42,
        ),
        {
            "n_clusters": [2, 3, 4, 5],
        },
    ),
    (
        "bisecting k-means",
        BisectingKMeans(
            random_state=42,
        ),
        {
            "n_clusters": [2, 3, 4, 5],
            "bisecting_strategy": [
                "biggest_inertia",
                "largest_cluster",
            ],
        },
    ),
    (
        "birch",
        Birch(),
        {
            "n_clusters": [2, 3, 4, 5],
            "threshold": [0.1, 0.3, 0.5],
        },
    ),
]

In [33]:
from sklearn.model_selection import ParameterGrid

In [ ]:
# Avaliação dos modelos em amostra controlada

TAMANHO_AMOSTRA = min(5_000, len(X_ref))
rng = np.random.default_rng(42)
indices = rng.choice(
    len(X_ref),
    size=TAMANHO_AMOSTRA,
    replace=False,
)

X_modelo = X_ref.iloc[indices].to_numpy(dtype=float)

resultados = []

for model_name, model_obj, model_params in models:
    for params in ParameterGrid(model_params):
        inicio = time.time()

        modelo = model_obj.set_params(**params)
        modelo.fit(X_modelo)
        labels = modelo.labels_

        silhueta = silhouette_score(
            X_modelo,
            labels,
            metric="euclidean",
        )

        resultados.append(
            {
                "modelo": model_name,
                "parametros": params,
                "silhueta": silhueta,
                "grupos": len(np.unique(labels)),
                "tempo": time.time() - inicio,
            }
        )

resultados = pd.DataFrame(resultados).sort_values(
    "silhueta",
    ascending=False,
)

display(resultados)

,modelo,parametros,silhueta,grupos,tempo
8,mini batch k-means,{'n_clusters': 2},0.124943,2,1.586098
0,k-means,"{'init': 'k-means++', 'n_clusters': 2}",0.124618,2,2.488572
4,k-means,"{'init': 'random', 'n_clusters': 2}",0.124618,2,1.849045
7,k-means,"{'init': 'random', 'n_clusters': 5}",0.124263,5,1.636594
3,k-means,"{'init': 'k-means++', 'n_clusters': 5}",0.124174,5,3.226920
11,mini batch k-means,{'n_clusters': 5},0.123240,5,1.642509
15,bisecting k-means,"{'bisecting_strategy': 'biggest_inertia', 'n_c...",0.115738,5,1.355387
19,bisecting k-means,"{'bisecting_strategy': 'largest_cluster', 'n_c...",0.115738,5,1.519344
13,bisecting k-means,"{'bisecting_strategy': 'biggest_inertia', 'n_c...",0.110517,3,1.651453
26,birch,"{'n_clusters': 4, 'threshold': 0.1}",0.110056,4,8.962976


In [35]:
# Selecionando o melhor modelo

melhor = resultados.iloc[0]

winner = melhor["modelo"]
best_params = melhor["parametros"]

print(f"Melhor modelo: {winner}")
print(f"Parâmetros: {best_params}")
print(f"Silhueta: {melhor['silhueta']:.3f}")
print(f"Grupos: {int(melhor['grupos'])}")

Melhor modelo: mini batch k-means
Parâmetros: {'n_clusters': 2}
Silhueta: 0.125
Grupos: 2


In [36]:
# Treinando o melhor modelo na matriz completa

model_name, model, _ = [
    item for item in models
    if item[0] == winner
][0]

modelo_final = model.set_params(**best_params)
modelo_final.fit(X_ref.to_numpy(dtype=float))

labels = modelo_final.labels_

X_t = X_ref.to_numpy(dtype=float)

display(
    Markdown(
        f"Modelo final: **{model_name}**  \n"
        f"Silhueta: **{silhouette_score(X_t, labels):.3f}**  \n"
        f"Grupos formados: **{len(np.unique(labels))}**"
    )
)

Modelo final: **mini batch k-means**  
Silhueta: **0.100**  
Grupos formados: **2**

In [37]:
# Perfil dos segmentos de reservas hoteleiras

perfil = (
    df.assign(segmento=labels)
    .groupby("segmento")
    .agg(
        reservas=("hotel", "size"),
        taxa_cancelamento=("is_canceled", "mean"),
        lead_time_mediano=("lead_time", "median"),
        noites_mediana=("total_nights", "median"),
        hospedes_mediana=("total_guests", "median"),
        adr_mediana=("adr", "median"),
        solicitacoes_mediana=("total_of_special_requests", "median"),
    )
    .assign(
        participacao=lambda tabela: tabela["reservas"] / len(df)
    )
    .sort_values("taxa_cancelamento", ascending=False)
)

display(Markdown("### Perfil dos segmentos"))
display(
    perfil.style.format(
        {
            "taxa_cancelamento": "{:.1%}",
            "participacao": "{:.1%}",
            "lead_time_mediano": "{:.0f}",
            "noites_mediana": "{:.0f}",
            "hospedes_mediana": "{:.0f}",
            "adr_mediana": "{:.2f}",
            "solicitacoes_mediana": "{:.0f}",
        }
    )
)

### Perfil dos segmentos

,reservas,taxa_cancelamento,lead_time_mediano,noites_mediana,hospedes_mediana,adr_mediana,solicitacoes_mediana,participacao
segmento,,,,,,,,
1,46819,47.3%,103,2,2,80.00,0,39.2%
0,72571,30.4%,54,3,2,106.59,1,60.8%


In [38]:
# Salvando o modelo bonus de hotelaria

PASTA_MODELOS = RAIZ / "models"
PASTA_PROCESSADOS = RAIZ / "data" / "processed"

PASTA_MODELOS.mkdir(parents=True, exist_ok=True)
PASTA_PROCESSADOS.mkdir(parents=True, exist_ok=True)

nome_modelo = (
    winner
    .replace(" ", "_")
    .replace("-", "_")
)

modelo_path = PASTA_MODELOS / f"model_bonus_hotel_{nome_modelo}.joblib"

joblib.dump(modelo_final, modelo_path)

pd.DataFrame(
    {
        "segmento": labels,
    }
).to_csv(
    PASTA_PROCESSADOS / "segmentos_bonus.csv",
    sep=";",
    index=False,
)

perfil.to_csv(
    PASTA_PROCESSADOS / "perfil_segmentos_bonus.csv",
    sep=";",
    encoding="utf-8",
)

display(
    Markdown(
        f"Modelo salvo em `{modelo_path}`  \n"
        "Rótulos salvos em `data/processed/segmentos_bonus.csv`  \n"
        "Perfil salvo em `data/processed/perfil_segmentos_bonus.csv`"
    )
)

Modelo salvo em `/home/luan/Área de trabalho/projeto-hotel-clientes/projeto-exemplo-mercado-imobiliario-main/models/model_bonus_hotel_mini_batch_k_means.joblib`  
Rótulos salvos em `data/processed/segmentos_bonus.csv`  
Perfil salvo em `data/processed/perfil_segmentos_bonus.csv`

# Visualização das reservas segmentadas

In [39]:
reservas_segmentadas = (
    df
    .assign(segmento=labels)
)

reservas_segmentadas.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,total_nights,total_guests,room_changed,has_special_requests,segmento
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0,0,Check-Out,2015-07-01,2015-07-01,0,2.0,0,0,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0,0,Check-Out,2015-07-01,2015-07-01,0,2.0,0,0,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,0,Check-Out,2015-07-02,2015-07-01,1,1.0,1,0,0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,0,Check-Out,2015-07-02,2015-07-01,1,1.0,0,0,1
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,1,Check-Out,2015-07-03,2015-07-01,2,2.0,0,1,0
